# 🩺 First Aid Chatbot

Run the cells in order (1, 2, 2B, 3 ... 8). After Cell 1, do **Runtime > Restart session**, then continue from Cell 2.

Save your private values in **Colab Secrets** (key icon, left sidebar, Notebook access ON): `GEMINI_API_KEY`, `NGROK_AUTHTOKEN`, `CHATBOT_PASSWORD`, `CHATBOT_API_KEY`. Optional: `NGROK_DOMAIN`, `DATA_CSV_URL`, `MODEL_URL`. See the README for details.

> Not medical advice. In a life-threatening emergency, call emergency services first.

In [ ]:
# %% CELL 1 - Install (run once, takes ~1-2 minutes)
!pip -q install google-genai "gradio>=5,<6" sentence-transformers pandas gTTS transformers accelerate pillow scikit-learn==1.6.1
# The red "dependency resolver" warnings are harmless - ignore them.
# If a later cell says "ModuleNotFoundError": Runtime > Restart session, then run from CELL 2.
# scikit-learn is pinned to 1.6.1 because YOUR saved model was trained with it; a different
# version can refuse to load the file or warn. Always restart the session after this cell.


In [ ]:
# %% CELL 2 - Settings and YOUR DATA (CSV)
import os, glob, re
import pandas as pd

DATA_DIR = "/content/firstaid_data"
# Preferred Gemini models, tried in order (CELL 5 uses the first one that works with your key)
MODEL_CANDIDATES = ["gemini-3.5-flash", "gemini-3.5-flash-lite", "gemini-flash-latest", "gemini-2.5-flash"]  # CELL 5 also auto-discovers the models your key can use
EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"   # multilingual model, used here for English meaning search
MIN_SIM = 0.35                       # below this similarity the bot says "not in my data" (raise = stricter)
EMERGENCY_LABELS = {"cardiac_arrest", "choking"}   # topics that ALWAYS trigger the "call emergency" banner

# Edit for your country. Egypt: ambulance 123, police 122, fire 180
EMERGENCY_TEXT = "Call emergency services NOW (Egypt: ambulance 123, police 122, fire 180; or 112)."

os.makedirs(DATA_DIR, exist_ok=True)

def quiet_secret(name):
    """Colab Secret or environment variable. Returns "" if it is not set. Never printed."""
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if isinstance(v, str) and v.strip():
            return v.strip()
    except Exception:
        pass
    return os.environ.get(name, "").strip()

# Your dataset: downloaded from the private link you saved in the secret DATA_CSV_URL.
# No secret saved? Then the upload box below appears instead.
import urllib.request
_CSV_URL = quiet_secret("DATA_CSV_URL")
if _CSV_URL:
    try:
        urllib.request.urlretrieve(_CSV_URL, os.path.join(DATA_DIR, "first_aid_clean.csv"))
    except Exception as e:
        print("Auto-download failed:", str(e)[:150], "- you can upload the CSV below instead.")


def find_csv():
    return sorted(glob.glob(f"{DATA_DIR}/*.csv"))

if not find_csv():
    print("Upload your data file now (first_aid_clean.csv):")
    from google.colab import files as colab_files
    uploaded = colab_files.upload()
    for name, data in uploaded.items():
        if name.lower().endswith(".csv"):
            with open(os.path.join(DATA_DIR, name), "wb") as f:
                f.write(data)

csvs = find_csv()
if not csvs:
    raise SystemExit("No .csv file found. Run this cell again and choose first_aid_clean.csv.")

df = pd.read_csv(csvs[0], encoding="utf-8-sig")
df.columns = [str(c).strip().lower() for c in df.columns]
missing = {"text", "label", "response"} - set(df.columns)
if missing:
    raise SystemExit(f"Your CSV needs the columns text, label, response. Missing: {sorted(missing)}. "
                     f"Columns found: {list(df.columns)}")

df = df.dropna(subset=["text", "label", "response"]).copy()
for c in ("text", "label", "response"):
    df[c] = df[c].astype(str).str.strip()
df = df[(df.text != "") & (df.label != "") & (df.response != "")].drop_duplicates(subset="text").reset_index(drop=True)
if df.empty:
    raise SystemExit("Your CSV has no usable rows.")

ex_texts = df["text"].tolist()          # example situations people describe
ex_labels = df["label"].tolist()        # the topic of each example

# One set of instructions per topic
PROTOCOLS = {}
for lab, grp in df.groupby("label"):
    PROTOCOLS[lab] = "\n".join(dict.fromkeys(grp["response"].tolist()))   # unique responses, original order

def pretty(label):
    return label.replace("_", " ")

TOPICS = ", ".join(pretty(l) for l in PROTOCOLS)
print(f"Loaded {os.path.basename(csvs[0])}: {len(df)} example situations, {len(PROTOCOLS)} topics")
print(df.label.value_counts().to_string())


In [ ]:
# %% CELL 2B - YOUR trained model (downloaded from your private link, or uploaded by you)
import joblib, os

MODEL_STATUS = "warning: trained model NOT loaded - bot is using meaning search only"

# The link lives in the Colab Secret MODEL_URL (not in this code). No secret? You are asked to upload the .pkl file.
# Only load model files you made yourself: a pickle from a stranger can run code.
import urllib.request as _ul
_MODEL_PATH = os.path.join(DATA_DIR, "firstaid_model.pkl")
if not os.path.exists(_MODEL_PATH):
    _MODEL_URL = quiet_secret("MODEL_URL")
    if _MODEL_URL:
        try:
            _ul.urlretrieve(_MODEL_URL, _MODEL_PATH)
        except Exception as e:
            print("Model download failed:", str(e)[:150])
    else:
        print("No MODEL_URL secret saved. Upload your trained model file (firstaid_model.pkl), or cancel to skip:")
        try:
            from google.colab import files as _cf
            for _n, _d in _cf.upload().items():
                if _n.lower().endswith(".pkl"):
                    with open(_MODEL_PATH, "wb") as _f:
                        _f.write(_d)
        except Exception as e:
            print("No model uploaded:", str(e)[:150])

clf_model, clf_answers = None, {}
try:
    bundle = joblib.load(_MODEL_PATH)
    clf_model = bundle["model"]
    clf_answers = bundle.get("answers", {})
    known = [str(c) for c in clf_model.classes_]
    missing = [c for c in known if c not in PROTOCOLS]
    MODEL_STATUS = "your trained classifier answers clear English text; meaning search covers unsure cases (English only)"
    print(f"OK - loaded your trained model: it knows {len(known)} topics -> {known}")
    if missing:
        print(f"Note: {missing} are not in the CSV topics; the meaning search will handle those instead.")
except Exception as e:
    clf_model = None
    print("=" * 70)
    print("WARNING: your trained model did NOT load. The bot will run on meaning search")
    print("only, and the app header will say so. Error:", str(e)[:300])
    print("If it mentions a version or unpickling problem: rerun CELL 1, then")
    print("Runtime > Restart session, then run from CELL 2 again.")
    print("=" * 70)


In [ ]:
# %% CELL 3 - Teach the bot to recognise situations (embeddings)
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBED_MODEL)

def embed(texts, bs=64):
    v = embedder.encode(list(texts), batch_size=bs, normalize_embeddings=True, show_progress_bar=False)
    return np.asarray(v, dtype="float32")

E = embed(ex_texts)

def classify(query, k=7):
    """Find the k most similar example situations and vote for a topic.
    Returns (best_label, confidence, ranked) where confidence = similarity of the closest example."""
    q = embed([query])[0]
    sims = E @ q
    k = min(k, len(ex_texts))
    top = np.argsort(-sims)[:k]
    votes, best_for = {}, {}
    for i in top:
        lab, s = ex_labels[i], float(sims[i])
        votes[lab] = votes.get(lab, 0.0) + max(s, 0.0)
        best_for[lab] = max(best_for.get(lab, -1.0), s)
    ranked = sorted(votes, key=lambda l: -votes[l])
    return ranked[0], float(sims[top[0]]), ranked

# Backup keywords (English). Used only when the embedding match is weak.
TOPIC_KEYWORDS = {
    "burn": [r"\bburn", r"scald", r"boiling", r"hot (oil|water|soup|tea)", r"sunburn"],
    "bleeding": [r"bleed", r"blood", r"deep cut", r"nosebleed"],
    "choking": [r"chok", r"cannot (speak|talk|breathe|make any sound)", r"can'?t (speak|talk)", r"stuck in (his|her|their|my) throat"],
    "allergic_reaction": [r"allerg", r"anaphyl", r"hives", r"sting"],
    "poisoning": [r"poison", r"overdose", r"swallowed", r"too many pills", r"bleach", r"unlabeled"],
    "cardiac_arrest": [r"no pulse", r"heartbeat", r"collaps", r"not breathing", r"cardiac", r"\bcpr\b", r"unresponsive"],
    "seizure": [r"seiz", r"convuls", r"epilep", r"foaming"],
    "fracture": [r"fractur", r"broken (bone|leg|arm|wrist|ankle)", r"\bbroke\b", r"(leg|arm|wrist|ankle) is broken"],
}
KEYWORD_RE = {lab: re.compile("|".join(p), re.IGNORECASE) for lab, p in TOPIC_KEYWORDS.items() if lab in PROTOCOLS}

def keyword_label(text):
    """The topic if EXACTLY one topic's keywords appear in the text, otherwise None."""
    hits = [lab for lab, rx in KEYWORD_RE.items() if rx.search(text or "")]
    return hits[0] if len(hits) == 1 else None

CLF_THRESHOLD = 0.5   # below this probability your trained model steps aside and the meaning search takes over

def clf_classify(text):
    """Your trained classifier (TF-IDF + LogisticRegression). Returns (label, probability).
    It learned from English examples."""
    if clf_model is None or not (text or "").strip():
        return None, 0.0
    try:
        probs = clf_model.predict_proba([text])[0]
        best = int(probs.argmax())
        label = str(clf_model.classes_[best])
        if label in PROTOCOLS:
            return label, float(probs[best])
    except Exception as e:
        print("Trained-model check skipped:", str(e)[:150])
    return None, 0.0

def match_topic(text, history):
    """Returns (label, confidence, ranked, source).
    Gate (measured on held-out queries): English needs ONE of: your trained model sure (>=0.60),
    a strong meaning match (>=0.68), or meaning match + keyword agreement (>=0.55).
    Anything weaker = "not in my data" instead of a confident wrong topic."""
    c_label, c_prob = clf_classify(text)
    label, conf, ranked = classify(text)
    kw = keyword_label(text)
    if c_prob >= 0.60:
        return c_label, c_prob, [c_label], "trained model"
    if conf >= 0.68:
        return label, conf, ranked, "meaning search"
    if kw and kw == label and conf >= 0.55:
        return label, conf, ranked, "meaning search + keyword"
    prev = [m["content"] for m in (history or []) if m.get("role") == "user" and isinstance(m.get("content"), str)]
    if prev:
        label2, conf2, ranked2 = classify(prev[-1] + " " + text)
        if conf2 >= 0.68:
            return label2, conf2, ranked2, "meaning search"
    return None, conf, ranked, "none"

# Quick self-check
S = E @ E.T
np.fill_diagonal(S, -1.0)
ok = 0
for i in range(len(ex_texts)):
    top = np.argsort(-S[i])[:7]
    v = {}
    for j in top:
        v[ex_labels[j]] = v.get(ex_labels[j], 0.0) + max(float(S[i, j]), 0.0)
    ok += (max(v, key=v.get) == ex_labels[i])
print(f"Self-check: {ok}/{len(ex_texts)} example situations matched their own topic ({100*ok/len(ex_texts):.1f}%)")


In [ ]:
# %% CELL 4 - Emergency red-flag words (checked BEFORE anything else)
RED_FLAGS = {
    "unresponsive / not breathing": [
        r"not breathing", r"stopped breathing", r"isn'?t breathing", r"unresponsive", r"unconscious",
        r"passed out", r"no pulse", r"no heartbeat", r"cardiac arrest", r"won'?t wake", r"not waking", r"collapsed",
    ],
    "severe bleeding": [
        r"bleeding (a lot|heavily|badly|won'?t stop)", r"severe bleeding", r"blood (everywhere|pouring|spurting)",
        r"arterial", r"won'?t stop bleeding", r"amputat", r"cut (off|his|her) (finger|hand|arm|leg)",
    ],
    "chest pain / heart attack": [
        r"chest pain", r"heart attack", r"pressure in (the |my |his |her )?chest", r"crushing chest",
    ],
    "stroke signs": [
        r"stroke", r"face (is )?droop", r"slurred speech", r"can'?t (move|lift) (his|her|their|one) (arm|side)",
        r"sudden (confusion|numbness|weakness)",
    ],
    "severe allergic reaction": [
        r"anaphyla", r"throat (is )?(closing|swelling)", r"swollen (tongue|throat|lips)", r"(lips|tongue|face) (and \w+ )?(is |are )?swelling",
    ],
    "choking": [
        r"choking", r"can'?t breathe", r"cannot breathe", r"cannot (make any )?sound", r"something stuck in (his|her|their|my) throat",
    ],
    "seizure / poisoning / drowning / burns": [
        r"seizure", r"convuls", r"overdose", r"poison", r"swallowed (bleach|pills|battery|chemical)", r"drown",
        r"electric shock", r"electrocut", r"severe burn", r"on fire",
    ],
}
RED_FLAG_RE = {label: re.compile("|".join(pats), re.IGNORECASE) for label, pats in RED_FLAGS.items()}

def detect_red_flags(text):
    return [label for label, rx in RED_FLAG_RE.items() if rx.search(text or "")]

def is_arabic(text):
    """True if the text contains Arabic letters (this bot answers in English only)."""
    return bool(re.search(r"[؀-ۿ]", text or ""))

SAFETY_PREFIX_EN = ("🚨 **{call}**\n"
                    "Put the phone on speaker so you can keep helping. **Check the area is safe first** "
                    "(traffic, fire, electric wires) so you do not get hurt too.\n\n")


In [ ]:
# %% CELL 5 - Gemini API key (reads it from Colab Secrets first, asks only if none is saved)
# Get a free key at https://aistudio.google.com/apikey  (the key is NOT stored in this code).
# SAVE IT ONCE: Colab left sidebar > key icon (Secrets) > Add new secret > Name: GEMINI_API_KEY > paste value > turn ON "Notebook access".
import os
from getpass import getpass

_alive = bool(globals().get("USE_AI")) and globals().get("client") is not None   # a key already working in this session
USE_AI = _alive
client = globals().get("client") if _alive else None
MODEL = globals().get("MODEL") if _alive else None
types = globals().get("types") if _alive else None

def discover_models(c):
    """Ask Google which models THIS key can use, so retired model names never break the bot."""
    skip = ("embedding", "tts", "image", "live", "audio", "imagen", "veo", "robotics", "computer-use",
            "learnlm", "gemma", "aqa", "native", "thinking-exp")
    found = []
    try:
        for m in c.models.list():
            name = (m.name or "").replace("models/", "")
            acts = getattr(m, "supported_actions", None) or []
            if "gemini" in name and "generateContent" in acts and not any(x in name for x in skip):
                found.append(name)
    except Exception as e:
        print("Could not list models:", str(e)[:150])
    # flash first (fast), then lite, then pro; stable names before previews
    found.sort(key=lambda n: ("pro" in n, "lite" in n, "preview" in n or "exp" in n, n))
    return found

def try_key(k):
    """Returns (ok, error_text). Sets the global client/MODEL when the key works."""
    global client, MODEL, USE_AI, types
    try:
        from google import genai
        from google.genai import types as _types
        types = _types
        c = genai.Client(api_key=k)
        names = list(dict.fromkeys(MODEL_CANDIDATES + discover_models(c)))[:10]
        errors = []
        for name in names:
            try:
                r = c.models.generate_content(model=name, contents="Reply with the single word OK.")
                if (r.text or "").strip():
                    client, MODEL, USE_AI = c, name, True
                    return True, ""
            except Exception as e:
                msg = str(e)
                if "API key not valid" in msg or "API_KEY_INVALID" in msg:
                    return False, "The API key itself is not valid. Copy it again from aistudio.google.com/apikey"
                errors.append(f"{name}: {msg[:90]}")
        return False, "No model worked. Details: " + " | ".join(errors[:3])
    except Exception as e:
        return False, str(e)[:300]

def saved_key():
    """Key saved in Colab Secrets (or an environment variable). Never printed."""
    for _name in ("GEMINI_API_KEY", "GEMINI_API_KAY"):    # second name = common typo, accepted as a fallback
        try:
            from google.colab import userdata
            k = userdata.get(_name)
            if k:
                return k
        except Exception:
            pass
    return os.environ.get("GEMINI_API_KEY", "") or os.environ.get("GOOGLE_API_KEY", "")

if not USE_AI:
    key = saved_key()
    if key:
        ok, err = try_key(key)
        if not ok:
            print(f"The saved key did not work ({err}).")
    if not USE_AI:
        for attempt in range(3):
            key = getpass("Paste your Gemini API key and press Enter (Enter alone = skip, no AI): ").strip()
            if not key:
                break                               # user chose to skip
            ok, err = try_key(key)
            if ok:
                print("Tip: save it once in Colab Secrets as GEMINI_API_KEY (key icon on the left) so it is not asked again.")
                break
            print(f"That key did not work ({err}). Try again ({attempt + 1}/3).")

if USE_AI:
    print(f"AI mode is ON (model: {MODEL}). Photos and voice are enabled.")
else:
    print("AI mode is OFF: photos and voice need a working key. Add GEMINI_API_KEY in Colab Secrets (key icon) or run this cell again to paste one.")
    print("(The bot still answers typed messages from your data.)")


In [ ]:
# %% CELL 6 - The chatbot brain (text + photo + voice)
import io, mimetypes
from PIL import Image

SYSTEM_PROMPT = f"""You are a first-aid guidance assistant for people who may be panicking and may have their hands busy.
Your knowledge comes ONLY from the REFERENCE the user message ends with. It covers these topics: {TOPICS}.

HOW TO ANSWER
- ALWAYS reply in English. If the user writes in any other language, reply only: "Sorry, I only answer in English."
- Use plain, everyday words. No medical jargon.
- Give only ONE or TWO actions at a time, as short commands that start with a bold verb: "**Press** hard on the wound."
- State the key "Do NOT" warning from the REFERENCE right next to the action it applies to.
- End most replies with ONE short yes/no question that moves things forward ("Are they awake and breathing?"). Never ask more than one question.
- Keep replies short. Instructions first, no long disclaimers.
- If the situation may be life-threatening, the FIRST line tells them to call emergency services, unless the message says they were already told.
- Adapt to adult, child or infant when the REFERENCE has different steps. If age matters and is unknown, ask.

ACCURACY
- Use ONLY the REFERENCE for medical steps and numbers. Never invent steps, doses or timings.
- If the situation is not covered (or the REFERENCE says NO TOPIC MATCHED), say "I don't have that in my data", give only the safe universal advice (call emergency services, keep the person still and calm, do not give food or drink) and say which topics you do cover.
- If the user asks what you cover or about the data itself, answer from the REFERENCE and the topic list.

LIMITS
- Never diagnose, never recommend or dose medicines that the REFERENCE does not mention, never instruct invasive procedures.
- If a photo is provided, describe only what matters for first aid and say you cannot judge it fully from a picture.
"""

def image_to_part(path):
    img = Image.open(path).convert("RGB")
    img.thumbnail((1280, 1280))
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=85)
    return types.Part.from_bytes(data=buf.getvalue(), mime_type="image/jpeg")

SUPPORTED_AUDIO = {".wav": "audio/wav", ".mp3": "audio/mp3", ".aiff": "audio/aiff",
                   ".aac": "audio/aac", ".ogg": "audio/ogg", ".flac": "audio/flac"}

def audio_to_part(path):
    ext = os.path.splitext(path)[1].lower()
    mime = SUPPORTED_AUDIO.get(ext) or mimetypes.guess_type(path)[0] or "audio/wav"
    with open(path, "rb") as f:
        return types.Part.from_bytes(data=f.read(), mime_type=mime)

def gemini_transcribe(audio_path):
    """Gemini listens to the recording and returns the words, in the language spoken."""
    r = client.models.generate_content(
        model=MODEL,
        contents=[audio_to_part(audio_path),
                  "Transcribe this audio exactly, in the language it is spoken. "
                  "Output ONLY the transcript, nothing else. If there is no speech, output nothing."])
    return (r.text or "").strip()

def gemini_describe_image(image_path):
    """Short English description of the emergency in the photo, used for red-flag + topic matching."""
    r = client.models.generate_content(
        model=MODEL,
        contents=[image_to_part(image_path),
                  "In ONE short sentence, describe the injury or emergency visible in this photo "
                  "(e.g. 'a burn on a forearm', 'heavy bleeding from a leg', 'person lying unconscious'). "
                  "If it is unrelated to first aid, say 'no first aid situation'."])
    d = (r.text or "").strip()
    return "" if "no first aid situation" in d.lower() else d

def clean_history(history, max_turns=10):
    """Keep only valid user/assistant text messages, alternating, starting with 'user'."""
    out = []
    for m in (history or [])[-max_turns * 2:]:
        role, content = m.get("role"), m.get("content")
        if role not in ("user", "assistant") or not isinstance(content, str) or not content.strip():
            continue
        if not out and role != "user":
            continue
        if out and out[-1]["role"] == role:
            out[-1] = {"role": role, "content": content}
        else:
            out.append({"role": role, "content": content})
    if out and out[-1]["role"] == "user":     # must end with an assistant turn before the new user turn
        out = out[:-1]
    return out

def reference_text(best_label):
    parts = []
    if best_label is None:
        parts.append("NO TOPIC MATCHED the user's message.")
    for lab, resp in PROTOCOLS.items():
        tag = "  <-- BEST MATCH for the user's message" if lab == best_label else ""
        parts.append(f"[Topic: {pretty(lab)}]{tag}\n{resp}")
    return "\n\n".join(parts)

def format_protocol(label):
    """Data-only mode: show the instructions from your CSV as short bullet points."""
    sents = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", PROTOCOLS[label].strip())
    lines = []
    for s in sents:
        s = s.strip()
        if not s:
            continue
        if re.match(r"(?i)^(do not|don't|never)\b", s):
            lines.append(f"- ⚠️ **{s}**")
        else:
            lines.append(f"- {s}")
    return f"**{pretty(label).capitalize()}**\n" + "\n".join(lines)

def no_match_text(text):
    return ("I don't have that situation in my data. Safe steps for any emergency:\n"
            "- **Call emergency services** if the person is badly hurt, confused, or not breathing normally.\n"
            "- **Keep** the person still and calm. **Do not** give food or drink.\n\n"
            f"I can help with: {TOPICS}.")

def generate(contents, max_tokens=1024):
    """Call Gemini. First try with 'thinking' switched off (faster); if the model rejects that, retry plainly."""
    base = dict(system_instruction=SYSTEM_PROMPT, max_output_tokens=max_tokens, temperature=0.2)
    try:
        cfg = types.GenerateContentConfig(**base, thinking_config=types.ThinkingConfig(thinking_budget=0))
        return client.models.generate_content(model=MODEL, contents=contents, config=cfg)
    except Exception:
        cfg = types.GenerateContentConfig(**base)
        return client.models.generate_content(model=MODEL, contents=contents, config=cfg)

def ask_ai(text, image_path, history, label, flags, emergency):
    note = ""
    if emergency:
        note = ("\n\nNOTE: The user has ALREADY been told to call emergency services and check "
                "scene safety, so do not repeat that. Go straight to the first one or two care steps.")
    parts = []
    if image_path:
        parts.append(image_to_part(image_path))
    parts.append(types.Part.from_text(text=f"{text}\n\n---\nREFERENCE:\n{reference_text(label)}{note}"))

    contents = []
    for m in clean_history(history):
        role = "user" if m["role"] == "user" else "model"
        contents.append(types.Content(role=role, parts=[types.Part.from_text(text=m["content"])]))
    contents.append(types.Content(role="user", parts=parts))

    resp = generate(contents)
    return (resp.text or "").strip()      # empty if blocked -> the caller falls back to your data

OUT_OF_LANE_EN = ("🚫 **That is out of my lane.** I am a first-aid assistant, so I can only help with first-aid "
                  "situations (I cover: {topics}).\n\nIf someone is hurt or in danger, tell me what happened, "
                  "send a photo, or speak to me.")
ENGLISH_ONLY_MSG = ("🚫 **Sorry, I only answer in English.** Please write or say your question in English "
                    "and I will help.")

SMALLTALK_RE = re.compile(r"^\s*(hi|hello|hey|yo|good (morning|evening|afternoon)|how are you|what'?s up|thanks?|thank you|"
                          r"who are you)\b", re.IGNORECASE)

def out_of_lane(text):
    return OUT_OF_LANE_EN.format(topics=TOPICS)

def looks_first_aid(text, history):
    """True if the message is about an injury / medical emergency (or a follow-up to one).
    Used only when no topic matched. Never raises: on any doubt it returns True (safe default)."""
    if not USE_AI:
        return not SMALLTALK_RE.search(text or "")          # no AI: only block obvious small talk
    try:
        ctx = ""
        last_bot = [m.get("content", "") for m in (history or []) if m.get("role") == "assistant"]
        if last_bot:
            ctx = f"Previous assistant message: {str(last_bot[-1])[:300]}\n"
        r = client.models.generate_content(
            model=MODEL,
            contents=(f"{ctx}User message: {text}\n\n"
                      "Is the user message about a first-aid situation, an injury, a medical emergency, or a "
                      "follow-up answer to one? Greetings, small talk, thanks and any unrelated topic are NO. "
                      "Answer with exactly one word: YES or NO."))
        return not re.match(r"\W*NO\b", (r.text or "YES").strip().upper())
    except Exception as e:
        print("Topic check failed, continuing normally:", str(e)[:150])
        return True

def gemini_photo_topic(image_path, text=""):
    """Gemini LOOKS at the photo and picks the matching topic from your data (or None).
    This is more reliable than matching a text description against the CSV examples."""
    names = [pretty(l) for l in PROTOCOLS]
    r = client.models.generate_content(
        model=MODEL,
        contents=[image_to_part(image_path),
                  f"Look carefully at this photo. Which ONE of these first-aid topics does it show: {', '.join(names)}? "
                  f"Example: a hand with red or blistered skin from heat = burn. "
                  f"{('The user also wrote: ' + text) if text else ''}\n"
                  "If the photo shows none of these topics, answer NONE. Answer with ONLY the topic name."])
    ans = re.sub(r"[^a-z ]", " ", (r.text or "").lower()).split()
    ans = " ".join(ans)
    for lab in PROTOCOLS:                       # exact topic name first, then contained
        if ans == pretty(lab).lower():
            return lab
    for lab in PROTOCOLS:
        if pretty(lab).lower() in ans:
            return lab
    return None

def chatbot_reply(text, image_path=None, history=None):
    """Returns (spoken_text, display_text). Handles text, photo, voice transcript, or a mix."""
    match_text = text or ""
    if is_arabic(text):                           # English only: refuse Arabic questions, in English
        return ENGLISH_ONLY_MSG, ENGLISH_ONLY_MSG
    photo_checked = False
    if image_path and USE_AI:                     # the photo also feeds emergency detection + topic matching
        try:
            desc = gemini_describe_image(image_path)
            photo_checked = True
            if desc:
                match_text = f"{match_text} {desc}".strip()
        except Exception as e:
            print("Photo description failed:", str(e)[:200])

    flags = detect_red_flags(match_text)
    from_photo = False
    if image_path and USE_AI:
        # topic comes from Gemini LOOKING at the photo (not from matching a description against the CSV)
        t_label, t_conf, t_ranked, t_source = match_topic(text, history) if (text or "").strip() else (None, 0.0, [], "")
        p_label = None
        try:
            p_label = gemini_photo_topic(image_path, text)
        except Exception as e:
            print("Photo topic failed:", str(e)[:200])
        if p_label:                      # what Gemini SEES in the photo always wins over vague typed words
            label, conf, ranked, from_photo, source = p_label, 1.0, [p_label], True, "photo"
        else:
            label, conf, ranked, source = t_label, t_conf, t_ranked, t_source
    else:
        label, conf, ranked, source = match_topic(match_text, history) if match_text else (None, 0.0, [], "")
    # Out-of-lane guard: nothing matched and no emergency words -> politely refuse (never an error)
    if not flags and label is None:
        if image_path and not match_text.strip() and photo_checked:
            msg = out_of_lane(text)
            return msg, msg
        if match_text.strip() and not looks_first_aid(match_text, history):
            msg = out_of_lane(text)
            return msg, msg

    emergency = bool(flags) or (label in EMERGENCY_LABELS)
    # show the big "call emergency" banner once per emergency, not on every follow-up message
    already_shown = any(m.get("role") == "assistant" and "🚨" in str(m.get("content", "")) for m in (history or []))

    prefix = ""
    if emergency and not already_shown:
        prefix = SAFETY_PREFIX_EN.format(call=EMERGENCY_TEXT)

    body = ""
    if USE_AI:
        try:
            body = ask_ai(text or "(photo only, please look at it)", image_path, history, label, flags, emergency)
        except Exception as e:
            print("AI error, using data-only answer:", str(e)[:200])
            body = ""
    if not body:   # data-only mode, or the AI failed: never leave the user without instructions
        body = format_protocol(label) if label else no_match_text(text)
        if image_path and not USE_AI:
            body += "\n\n_(Photos need AI mode.)_"

    spoken = prefix + body
    if label:
        footer = (f"\n\n_matched topic: {pretty(label)} (from photo)_" if from_photo
                  else f"\n\n_matched topic: {pretty(label)} via {source} · confidence {conf:.2f}_")
    elif not history:
        footer = f"\n\n_no matching topic · closest similarity {conf:.2f}_"
    else:
        footer = ""
    return spoken, spoken + footer

# quick self-test so problems show up here, not in the app
print(chatbot_reply("my friend is choking on food and cannot talk")[1][:500])


In [ ]:
# %% CELL 7 - Voice (Gemini transcription, Whisper backup, spoken replies)
import re, tempfile
from gtts import gTTS

_asr = None
def whisper_transcribe(audio_path):
    global _asr
    if _asr is None:
        import torch
        from transformers import pipeline
        _asr = pipeline("automatic-speech-recognition", model="openai/whisper-small",
                        device=0 if torch.cuda.is_available() else -1)
    return _asr(audio_path, generate_kwargs={"task": "transcribe"})["text"].strip()

RECORD_DIR = "/content/firstaid_data/recordings"     # every voice recording is saved here
os.makedirs(RECORD_DIR, exist_ok=True)

def save_and_convert(audio_path):
    """Saves a copy of the recording in RECORD_DIR and returns a clean 16 kHz mono WAV (what Gemini likes)."""
    import shutil, subprocess, time
    stamp = time.strftime("%Y%m%d_%H%M%S")
    ext = os.path.splitext(audio_path)[1] or ".wav"
    saved = os.path.join(RECORD_DIR, f"rec_{stamp}{ext}")
    shutil.copy(audio_path, saved)
    wav = os.path.join(RECORD_DIR, f"rec_{stamp}_16k.wav")
    try:
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", saved, "-ar", "16000", "-ac", "1", wav],
                       check=True, capture_output=True)
        return wav
    except Exception as e:
        print("ffmpeg conversion failed, using the original file:", str(e)[:150])
        return saved

def transcribe(audio_path):
    """Gemini first (best for Arabic dialects); local Whisper if Gemini is off or fails."""
    audio_path = save_and_convert(audio_path)
    errors = []
    if USE_AI:
        try:
            t = gemini_transcribe(audio_path)
            if t:
                return t
            errors.append("Gemini heard no speech")
        except Exception as e:
            errors.append("Gemini: " + str(e)[:150])
            print("Gemini transcription failed, trying Whisper:", str(e)[:200])
    else:
        errors.append("AI mode is OFF (no working key)")
    try:
        t = whisper_transcribe(audio_path)
        if t:
            return t
        errors.append("Whisper heard no speech")
    except Exception as e:
        errors.append("Whisper: " + str(e)[:150])
    raise RuntimeError(" | ".join(errors))

def speak(text):
    try:
        clean = re.sub(r"[*_#`🚨⚠️\-]", " ", text)
        path = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3").name
        gTTS(clean, lang="en").save(path)
        return path
    except Exception as e:
        print("Text-to-speech failed:", e)
        return None


In [ ]:
# %% CELL 8 - Chat app + REST API (merged: UI, API and fixed address in ONE cell)
import gradio as gr, os, sys, subprocess
try:
    gr.close_all()                      # re-running this cell: free the old app first
except Exception:
    pass

def respond(text, image_path, audio_path, history, speak_reply):
    history = history or []
    text = (text or "").strip()
    heard = False
    if audio_path:                                  # the microphone recording is transcribed here
        stt_err = ""
        try:
            spoken_words = transcribe(audio_path)
        except Exception as e:
            print("Speech recognition failed:", e)
            spoken_words, stt_err = "", str(e)[:300]
        if spoken_words:
            text = (text + " " + spoken_words).strip()
            heard = True
        elif not text and not image_path:
            msg = ("I could not hear you clearly. Please record again, closer to the microphone.")
            if stt_err:
                msg += f"\n\n_(technical reason: {stt_err})_"
            return history + [{"role": "assistant", "content": msg}], "", None, None, None
    if not text and not image_path:
        return history, "", None, None, None
    try:
        spoken, shown = chatbot_reply(text, image_path, history)
    except Exception as e:
        print("Error:", e)
        spoken = shown = "Sorry, something went wrong. " + EMERGENCY_TEXT
    shown_user = (("🎤 " if heard else "") + (text or "(photo)")) + ("  📷" if image_path else "")
    history = history + [{"role": "user", "content": shown_user}, {"role": "assistant", "content": shown}]
    audio_out = speak(spoken) if speak_reply else None
    return history, "", None, None, audio_out

with gr.Blocks(title="First Aid Chatbot") as demo:
    gr.Markdown("## 🩺 First Aid Chatbot\n**In a life-threatening emergency, call emergency services first.** "
                f"First-aid guidance only (topics: {TOPICS}). Type, speak, or send a photo. **English only.**"
                f"\n\n**Engine:** {MODEL_STATUS}")
    chat = gr.Chatbot(height=420, type="messages")
    with gr.Row():
        txt = gr.Textbox(placeholder="Describe what happened (English only)...", scale=4)
        send = gr.Button("Send", variant="primary", scale=1)
    with gr.Row():
        img = gr.Image(sources=["upload", "webcam"], type="filepath", label="📷 Photo (take or upload)", height=200)
        mic = gr.Audio(sources=["microphone"], type="filepath", label="🎤 Press record, speak, then press stop")
        out_audio = gr.Audio(label="Spoken reply", autoplay=True)
    with gr.Row():
        perm_btn = gr.Button("🎤 Allow microphone (press this first)", variant="secondary")
        audio_file = gr.File(label="📁 Or upload a voice/audio file (any file type shown)", type="filepath")
    speak_cb = gr.Checkbox(label="Read replies aloud (hands-free)", value=False)
    clear = gr.Button("New emergency / clear")

    inputs = [txt, img, mic, chat, speak_cb]
    outputs = [chat, txt, img, mic, out_audio]
    send.click(respond, inputs, outputs, api_name="chat")   # Gradio's own API: text + photo + voice (+ chat history)
    txt.submit(respond, inputs, outputs)
    mic.change(respond, inputs, outputs)          # runs as soon as a recording is ready
    clear.click(lambda: ([], "", None, None, None), None, outputs)

    # uploaded audio file -> same pipeline as the microphone
    def respond_file(text, image_path, file_path, history, speak_reply):
        out = respond(text, image_path, file_path, history, speak_reply)
        return (*out, None)          # also empties the upload box
    audio_file.change(respond_file, [txt, img, audio_file, chat, speak_cb], outputs + [audio_file])

    # microphone permission: the browser asks for it when the page opens and when you press the button
    ASK_MIC_JS = """
    async () => {
      try {
        const s = await navigator.mediaDevices.getUserMedia({audio: true});
        s.getTracks().forEach(t => t.stop());
        alert("✅ Microphone allowed. You can record now.");
      } catch (e) {
        alert("❌ Microphone blocked (" + e.name + "). Click the lock icon next to the web address, " +
              "set Microphone to Allow, then reload. Open the public link, not the preview inside Colab.");
      }
    }"""
    ASK_MIC_ON_LOAD_JS = """
    async () => {
      try {
        const s = await navigator.mediaDevices.getUserMedia({audio: true});
        s.getTracks().forEach(t => t.stop());
      } catch (e) { console.log("Microphone permission not granted yet:", e.name); }
    }"""
    # Gradio API: plain text question -> answer text (api_name "ask")
    def ask_api(text: str) -> str:
        spoken, shown = chatbot_reply((text or "").strip(), None, [])
        return shown
    gr.api(ask_api, api_name="ask")

    perm_btn.click(None, None, None, js=ASK_MIC_JS)
    demo.load(None, None, None, js=ASK_MIC_ON_LOAD_JS)

# ======================= PART 2: REST API + UI on one fixed address =======================
import json, shutil, tempfile, threading, time
import uvicorn
from typing import Optional, List, Dict
from fastapi import FastAPI, Header, HTTPException, UploadFile, File, Form
from pydantic import BaseModel

def _secret(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if isinstance(v, str) and v.strip():
            return v
    except Exception:
        pass
    return os.environ.get(name, "")

from getpass import getpass
def _secret_or_ask(name, label):
    """Use the Colab Secret if it exists, otherwise ask you to type it in a hidden box below the cell."""
    v = _secret(name)
    for _ in range(3):
        if isinstance(v, str) and v.strip():
            return v.strip()
        v = getpass(f"{label} (paste/type it and press Enter): ")
    return ""

# Your fixed ngrok address is a Colab Secret (NGROK_DOMAIN), not part of the code.
# Optional: if it is not saved, ngrok gives you a random address instead.
FIXED_HOST   = str(_secret("NGROK_DOMAIN") or "").strip().replace("https://", "").strip("/")
NGROK_TOKEN = _secret_or_ask("NGROK_AUTHTOKEN", "1/3 ngrok authtoken (from dashboard.ngrok.com > Your Authtoken)")
UI_PASSWORD = _secret_or_ask("CHATBOT_PASSWORD", "2/3 Choose a password for the web page (you pick it)")
API_KEY     = _secret_or_ask("CHATBOT_API_KEY", "3/3 API key for the app (any long random text)")
if not (NGROK_TOKEN and UI_PASSWORD and API_KEY):
    raise SystemExit("STOPPED (nothing was started). One of the three values was left empty. Run this cell again "
                     "and fill all three boxes (or save them in Colab Secrets with Notebook access ON).")

api = FastAPI(title="First Aid Chatbot API")

def check_key(k):
    if k != API_KEY:
        raise HTTPException(status_code=401, detail="Invalid or missing X-API-Key")

class AskIn(BaseModel):
    text: str
    history: Optional[List[Dict[str, str]]] = None   # [{"role":"user"|"assistant","content":"..."}]

@api.get("/api/health")
def health():
    return {"ok": True, "ai": bool(USE_AI), "model": MODEL}

@api.post("/api/ask")
def api_ask(body: AskIn, x_api_key: str = Header(None)):
    check_key(x_api_key)
    if not body.text.strip():
        raise HTTPException(400, "text is empty")
    spoken, shown = chatbot_reply(body.text.strip(), None, body.history or [])
    return {"reply": shown, "spoken": spoken}

@api.post("/api/chat")
def api_chat(text: str = Form(""), history: str = Form("[]"),
             image: Optional[UploadFile] = File(None),
             audio: Optional[UploadFile] = File(None),
             x_api_key: str = Header(None)):
    check_key(x_api_key)
    try:
        hist = json.loads(history or "[]")
    except Exception:
        raise HTTPException(400, "history must be a JSON list")

    def save(up):
        suffix = os.path.splitext(up.filename or "")[1] or ".bin"
        tmp = tempfile.NamedTemporaryFile(delete=False, suffix=suffix)
        shutil.copyfileobj(up.file, tmp); tmp.close()
        return tmp.name

    img_path = save(image) if image else None
    transcript = ""
    if audio:
        try:
            transcript = transcribe(save(audio))
        except Exception as e:
            if not text.strip() and not img_path:
                raise HTTPException(422, f"Could not transcribe audio: {str(e)[:200]}")
    msg = (text.strip() + " " + transcript).strip()
    if not msg and not img_path:
        raise HTTPException(400, "Send text, image or audio")
    spoken, shown = chatbot_reply(msg or "(photo)", img_path, hist)
    return {"reply": shown, "spoken": spoken, "transcript": transcript}

# The Gradio UI stays available at "/"
app = gr.mount_gradio_app(api, demo, path="/", auth=("firstaid", UI_PASSWORD))

# Stop an old server if you re-run the cell
if globals().get("_server") is not None:
    _server.should_exit = True
    time.sleep(2)

subprocess.run([sys.executable, "-m", "pip", "-q", "install", "pyngrok"], check=True)
from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_TOKEN
for _t in ngrok.get_tunnels():
    ngrok.disconnect(_t.public_url)

_server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=7860, log_level="warning"))
threading.Thread(target=_server.run, daemon=True).start()
time.sleep(3)
if FIXED_HOST:
    ngrok.connect(7860, "http", url=f"https://{FIXED_HOST}")
    PUBLIC_URL = f"https://{FIXED_HOST}"
else:
    PUBLIC_URL = ngrok.connect(7860, "http").public_url
print(f"UI:  {PUBLIC_URL}   (user: firstaid)")
print(f"API: {PUBLIC_URL}/api/ask  and  /api/chat   (header X-API-Key)")
